# KPI Extraction & Dependency Discovery (Standalone)

Self-contained Celonis notebook for **handover**. All logic lives in the cells below — **stdlib + pycelonis only**, no project Python scripts.

## Pipeline

```
Celonis Knowledge Model  →  kpis.json
PQL KPI("...") parsing   →  kpi_dependents.docx
Celonis Data Model FKs   →  sap_activity_tables_connectivity.json
```

## Output files (written to `OUTPUT_DIR`)

| File | Contents |
|------|----------|
| `kpis.json` | `[{kpi_id, kpi_name, pql}]` |
| `kpi_dependents.docx` | One Word section per KPI: dependents, paths, SAP & activity tables |
| `sap_activity_tables_connectivity.json` | Table join graph with column keys (array) |

**Run cells top-to-bottom.** Set `SKIP_CELONIS_EXTRACTION = True` to rebuild from cached JSON in `OFFLINE_DIR`.

## 0 - Prerequisites

Install once per environment (uncomment and run if needed):

In [2]:
%pip install --extra-index-url=https://pypi.celonis.cloud/ "pycelonis>=1.6.*"
%pip install pandas python-dotenv python-docx

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: 'pycelonis>=1.6.*': .* suffix can only be used with `==` or `!=` operators
    pycelonis>=1.6.*
             ~~~~~~^


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1 — Configuration

Fill in your Celonis connection details. Optionally load from a `.env` file in the same folder as this notebook.

| Variable | Description |
|----------|-------------|
| `CELONIS_URL` | Team URL, e.g. `https://your-team.eu-3.celonis.cloud` |
| `CELONIS_API_TOKEN` | API key from Celonis → Profile → API Keys |
| `CELONIS_KEY_TYPE` | Usually `USER_KEY` |
| `SPACE_ID` | Studio space UUID |
| `PACKAGE_ID` | Package UUID inside the space |
| `KNOWLEDGE_MODEL_ID` | Knowledge Model id for KPI extraction |
| `DATA_MODEL_ID` | Data model UUID for table FK connectivity |

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

# Load .env from cwd or parent (repo root)
try:
    from dotenv import load_dotenv

    for _env_dir in (Path.cwd(), Path.cwd().parent):
        _env_file = _env_dir / ".env"
        if _env_file.is_file():
            load_dotenv(_env_file)
            break
except ImportError:
    pass

# ── Celonis connection (defaults match project .env) ─────────────────────────
CELONIS_URL = os.getenv("CELONIS_URL", "https://abinbev-prod.eu-3.celonis.cloud")
CELONIS_API_TOKEN = os.getenv("CELONIS_API_TOKEN", "")  # set in .env — never commit tokens
CELONIS_KEY_TYPE = os.getenv("CELONIS_KEY_TYPE", "USER_KEY")

SPACE_ID = ""
PACKAGE_ID = ""
KNOWLEDGE_MODEL_ID = ""
DATA_MODEL_ID = ""

if not CELONIS_API_TOKEN:
    raise ValueError("CELONIS_API_TOKEN is missing — add it to .env or set the variable above.")

# ── Runtime switches ───────────────────────────────────────────────────────────
SKIP_CELONIS_EXTRACTION = False  # True = load JSON from OFFLINE_DIR instead

# ── Output paths (notebook-local) ─────────────────────────────────────────────
OUTPUT_DIR = Path(os.getenv("KPI_DISCOVERY_OUTPUT_DIR", "./kpi_discovery_output"))
OFFLINE_DIR = Path(os.getenv("KPI_DISCOVERY_OFFLINE_DIR", "./kpi_discovery_offline"))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Offline cache:    {OFFLINE_DIR.resolve()}")

Output directory: C:\Users\abcom\Desktop\ABI_Code_Migration_POC\kpi_discovery_output
Offline cache:    C:\Users\abcom\Desktop\ABI_Code_Migration_POC\kpi_discovery_offline


## 2 — Inline logic

PQL parsing, KPI dependency graph, SAP/activity table detection, and data-model FK grouping — all defined in this notebook.

In [4]:
import json
import re
from collections import defaultdict
from datetime import datetime, timezone
from typing import Any, Dict, Iterable, List, Optional, Set, Tuple

KPI_CALL_RE = re.compile(r"""(?ix)\bKPI\s*\(\s*(?:"([^"]+)"|'([^']+)')\s*\)""")
TABLE_COL_RE = re.compile(r'"([^"]+)"\s*\.\s*"([^"]+)"')
COUNT_TABLE_RE = re.compile(r'COUNT_TABLE\s*\(\s*"([^"]+)"\s*\)', re.I)

ACTIVITY_TABLE_PATTERNS = (
    re.compile(r"ACTIVIT", re.I),
    re.compile(r"^ACTIVITIES$", re.I),
    re.compile(r"_CEL_MERGED_ACTIVITIES", re.I),
)


def extract_kpi_refs(pql: str) -> List[str]:
    refs: List[str] = []
    for match in KPI_CALL_RE.finditer(pql or ""):
        ref = (match.group(1) or match.group(2) or "").strip()
        if ref:
            refs.append(ref)
    return refs


def is_activity_table(name: str) -> bool:
    return any(pattern.search(name) for pattern in ACTIVITY_TABLE_PATTERNS)


def extract_tables_from_pql(pql: str) -> Tuple[List[str], List[str]]:
    tables: Set[str] = set()
    for table, _col in TABLE_COL_RE.findall(pql or ""):
        tables.add(table.strip().upper())
    for table in COUNT_TABLE_RE.findall(pql or ""):
        tables.add(table.strip().upper())
    sap_tables: Set[str] = set()
    activity_tables: Set[str] = set()
    for table in tables:
        if is_activity_table(table):
            activity_tables.add(table)
        else:
            sap_tables.add(table)
    return sorted(sap_tables), sorted(activity_tables)


def build_kpi_graph(kpi_rows: List[Dict[str, Any]]) -> Tuple[Dict[str, List[str]], Dict[str, Dict[str, Any]]]:
    index: Dict[str, Dict[str, Any]] = {}
    for row in kpi_rows:
        if str(row.get("attribute_type", "KPI")).lower() != "kpi":
            continue
        kid = str(row.get("kpi_id", "") or "").strip()
        if kid:
            index[kid] = row

    graph: Dict[str, List[str]] = {}
    for kpi_id, row in index.items():
        pql = str(row.get("pql_formula", "") or "")
        embedded = row.get("referenced_kpi_ids")
        refs = embedded if isinstance(embedded, list) and embedded else extract_kpi_refs(pql)
        deps: List[str] = []
        seen: Set[str] = set()
        for ref in refs:
            ref = str(ref).strip()
            if ref in index and ref not in seen:
                seen.add(ref)
                deps.append(ref)
        graph[kpi_id] = deps
    return graph, index


def reachable_deps(start: str, graph: Dict[str, List[str]]) -> List[str]:
    seen: Set[str] = set()
    order: List[str] = []
    stack = [start]
    while stack:
        node = stack.pop()
        for dep in graph.get(node, []):
            if dep not in seen:
                seen.add(dep)
                order.append(dep)
                stack.append(dep)
    return order


def build_dependency_path_strings(start: str, graph: Dict[str, List[str]]) -> List[str]:
    paths: List[str] = []

    def dfs(node: str, chain: List[str], seen: Set[str]) -> None:
        if node in seen:
            paths.append(" -> ".join(chain + [f"[CYCLE:{node}]"]))
            return
        deps = graph.get(node, [])
        chain_with_node = chain + [node]
        if not deps:
            paths.append(" -> ".join(chain_with_node))
            return
        seen_next = set(seen) | {node}
        for dep in deps:
            dfs(dep, chain_with_node, seen_next)

    dfs(start, [], set())
    return paths


def build_table_connectivity(dm: Dict[str, Any]) -> List[Dict[str, Any]]:
    tables = dm.get("tables") or []
    id_to_name: Dict[str, str] = {}
    for table in tables:
        name = str(table.get("table_name", "") or "")
        if name:
            id_to_name[name] = name
        for key in ("table_id", "id"):
            tid = table.get(key)
            if tid and name:
                id_to_name[str(tid)] = name

    def resolve_name(value: str) -> str:
        value = str(value or "").strip()
        return id_to_name.get(value, value)

    edge_map: Dict[Tuple[str, str], Dict[str, Any]] = {}

    for fk in dm.get("foreign_keys") or []:
        src = resolve_name(
            fk.get("source_table_name")
            or fk.get("source_table")
            or fk.get("source_table_id")
            or ""
        )
        tgt = resolve_name(
            fk.get("target_table_name")
            or fk.get("target_table")
            or fk.get("target_table_id")
            or ""
        )
        if not src or not tgt:
            continue

        key = (src, tgt)
        if key not in edge_map:
            edge_map[key] = {
                "source": {"table": src, "columns": []},
                "connected_to": {"table": tgt, "columns": []},
            }

        mappings = fk.get("column_mappings")
        if isinstance(mappings, list) and mappings:
            for mapping in mappings:
                src_col = mapping.get("source_column_name") or mapping.get("source_column") or ""
                tgt_col = mapping.get("target_column_name") or mapping.get("target_column") or ""
                if src_col:
                    edge_map[key]["source"]["columns"].append(str(src_col))
                if tgt_col:
                    edge_map[key]["connected_to"]["columns"].append(str(tgt_col))
        else:
            src_col = fk.get("source_column_name") or fk.get("source_column") or ""
            tgt_col = fk.get("target_column_name") or fk.get("target_column") or ""
            if src_col:
                edge_map[key]["source"]["columns"].append(str(src_col))
            if tgt_col:
                edge_map[key]["connected_to"]["columns"].append(str(tgt_col))

    by_source: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
    all_tables: Set[str] = set()

    for table in tables:
        name = str(table.get("table_name", "") or "")
        if name:
            all_tables.add(name)

    for (src, tgt), connection in sorted(edge_map.items()):
        all_tables.add(src)
        all_tables.add(tgt)
        by_source[src].append(connection)

    return [
        {"table_name": table_name, "connections": by_source.get(table_name, [])}
        for table_name in sorted(all_tables)
    ]


def build_output_payloads(
    kpi_rows: List[Dict[str, Any]],
    dm_metadata: Optional[Dict[str, Any]],
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], List[Dict[str, Any]]]:
    graph, _index = build_kpi_graph(kpi_rows)

    kpis_json: List[Dict[str, Any]] = []
    dependents_json: List[Dict[str, Any]] = []

    for row in kpi_rows:
        if str(row.get("attribute_type", "KPI")).lower() != "kpi":
            continue
        kpi_id = str(row.get("kpi_id", "") or "").strip()
        if not kpi_id:
            continue

        kpi_name = str(row.get("name") or row.get("kpi_name") or kpi_id)
        pql = str(row.get("pql_formula") or row.get("pql") or "")

        kpis_json.append({"kpi_id": kpi_id, "kpi_name": kpi_name, "pql": pql})

        direct_deps = graph.get(kpi_id, [])
        transitive_deps = reachable_deps(kpi_id, graph) if direct_deps else []
        path_strings = build_dependency_path_strings(kpi_id, graph) if direct_deps else []
        sap_tables, activity_tables = extract_tables_from_pql(pql)

        dependents_json.append(
            {
                "kpi_id": kpi_id,
                "kpi_name": kpi_name,
                "has_dependents": "yes" if direct_deps else "no",
                "kpi_dependents": transitive_deps,
                "dependency_paths": path_strings,
                "sap_tables_used": sap_tables,
                "activity_tables_used": activity_tables,
            }
        )

    connectivity_json = build_table_connectivity(dm_metadata or {"tables": [], "foreign_keys": []})
    return kpis_json, dependents_json, connectivity_json


def write_kpi_dependents_word(rows: List[Dict[str, Any]], out_path: Path) -> None:
    """Write one Word document with a section for every KPI."""
    from docx import Document

    doc = Document()
    doc.add_heading("KPI Dependents Handover", 0)
    doc.add_paragraph(f"Generated (UTC): {datetime.now(timezone.utc).isoformat()}")
    doc.add_paragraph(f"Total KPIs: {len(rows)}")
    doc.add_paragraph("")

    def _add_bullets(label: str, items: List[str]) -> None:
        p = doc.add_paragraph()
        p.add_run(f"{label}:").bold = True
        if items:
            for item in items:
                doc.add_paragraph(str(item), style="List Bullet")
        else:
            doc.add_paragraph("(none)", style="List Bullet")

    for index, row in enumerate(rows, start=1):
        kpi_id = str(row.get("kpi_id", ""))
        kpi_name = str(row.get("kpi_name", kpi_id))
        doc.add_heading(f"{index}. {kpi_name}", level=1)

        id_p = doc.add_paragraph()
        id_p.add_run("KPI ID: ").bold = True
        id_p.add_run(kpi_id)

        dep_p = doc.add_paragraph()
        dep_p.add_run("Has dependents: ").bold = True
        dep_p.add_run(str(row.get("has_dependents", "no")))

        _add_bullets("KPI dependents", list(row.get("kpi_dependents") or []))
        _add_bullets("Dependency paths", list(row.get("dependency_paths") or []))
        _add_bullets("SAP tables used", list(row.get("sap_tables_used") or []))
        _add_bullets("Activity tables used", list(row.get("activity_tables_used") or []))

        if index < len(rows):
            doc.add_paragraph("")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    doc.save(out_path)


print("Inline helpers loaded.")

Inline helpers loaded.


## 3 — Celonis KM extraction helpers

Walks KPI / Filter / Record-attribute trees from the Knowledge Model (depth-first).

In [5]:
def _is_kpi_like(obj: Any) -> bool:
    return bool(getattr(obj, "pql", None) and getattr(obj, "id", None))


def _iter_child_kpi_nodes(parent: Any) -> Iterable[Any]:
    for attr in (
        "child_kpis", "children", "nested_kpis", "kpis", "sub_kpis",
        "items", "definitions", "nested_items", "parameters",
        "breakdowns", "aggregations",
    ):
        value = getattr(parent, attr, None)
        if value is None:
            continue
        if isinstance(value, dict):
            yield from (item for item in value.values() if item is not None)
            continue
        if isinstance(value, (list, tuple)):
            yield from (item for item in value if item is not None)
            continue
        if hasattr(value, "__iter__") and not isinstance(value, (str, bytes)):
            try:
                yield from (item for item in value if item is not None)
            except TypeError:
                pass


def _gather_nested_kpi_ids(roots: List[Any]) -> Set[str]:
    nested: Set[str] = set()

    def walk(obj: Any) -> None:
        for child in _iter_child_kpi_nodes(obj):
            if _is_kpi_like(child):
                cid = str(getattr(child, "id", "") or "")
                if cid:
                    nested.add(cid)
            walk(child)

    for root in roots:
        walk(root)
    return nested


def _row_from_kpi_object(obj: Any, attribute_type: str, record_id: str) -> Optional[Dict[str, Any]]:
    pql = getattr(obj, "pql", None)
    if not pql:
        return None
    kpi_id = str(getattr(obj, "id", "") or "")
    if not kpi_id:
        return None
    name = getattr(obj, "display_name", None) or getattr(obj, "name", None) or kpi_id
    pql_str = str(pql).strip()
    return {
        "kpi_id": kpi_id,
        "name": str(name),
        "pql_formula": pql_str,
        "attribute_type": attribute_type,
        "record_id": record_id,
        "referenced_kpi_ids": extract_kpi_refs(pql_str),
    }


def _collect_kpi_subtree(
    obj: Any, attribute_type: str, record_id: str, out_rows: List[Dict[str, Any]]
) -> None:
    if _is_kpi_like(obj):
        row = _row_from_kpi_object(obj, attribute_type, record_id)
        if row:
            out_rows.append(row)
    for child in _iter_child_kpi_nodes(obj):
        _collect_kpi_subtree(child, attribute_type, record_id, out_rows)


def extract_kpis_from_knowledge_model(km: Any) -> List[Dict[str, Any]]:
    content = km.get_content()
    if content is None:
        return []

    rows: List[Dict[str, Any]] = []

    kpis = list(content.kpis or [])
    nested_kpi_ids = _gather_nested_kpi_ids(kpis)
    for kpi in kpis:
        kid = str(getattr(kpi, "id", "") or "")
        if kid and kid in nested_kpi_ids:
            continue
        _collect_kpi_subtree(kpi, "KPI", "knowledge_model", rows)

    filters = list(content.filters or [])
    nested_filter_ids = _gather_nested_kpi_ids(filters)
    for flt in filters:
        fid = str(getattr(flt, "id", "") or "")
        if fid and fid in nested_filter_ids:
            continue
        _collect_kpi_subtree(flt, "Filter", "knowledge_model", rows)

    for record in list(content.records or []):
        record_id = str(getattr(record, "id", "unknown_record") or "unknown_record")
        attrs = list(record.attributes or [])
        nested_attr_ids = _gather_nested_kpi_ids(attrs)
        for attr in attrs:
            aid = str(getattr(attr, "id", "") or "")
            if aid and aid in nested_attr_ids:
                continue
            _collect_kpi_subtree(attr, "Attribute", record_id, rows)

    return [row for row in rows if str(row.get("attribute_type", "")).lower() == "kpi"]


def extract_data_model_metadata(celonis: Any, data_model_id: str) -> Dict[str, Any]:
    target_dm = None
    for pool in celonis.data_integration.get_data_pools():
        try:
            candidate = pool.get_data_model(data_model_id)
            if candidate:
                target_dm = candidate
                break
        except Exception:
            continue

    if target_dm is None:
        raise ValueError(f"Data model {data_model_id!r} not found in any pool")

    tables: List[Dict[str, Any]] = []
    id_to_name: Dict[str, str] = {}

    for table in target_dm.get_tables():
        table_name = str(getattr(table, "name", "") or "")
        table_id = str(getattr(table, "id", "") or "")
        columns = [str(c.name) for c in table.get_columns()]
        tables.append(
            {
                "table_id": table_id,
                "table_name": table_name,
                "columns": columns,
                "column_count": len(columns),
            }
        )
        if table_id and table_name:
            id_to_name[table_id] = table_name

    foreign_keys: List[Dict[str, Any]] = []
    grouped: Dict[str, Dict[str, Any]] = {}

    for fk in target_dm.get_foreign_keys():
        fk_id = str(getattr(fk, "id", "") or "")
        src_id = str(getattr(fk, "source_table_id", "") or "")
        tgt_id = str(getattr(fk, "target_table_id", "") or "")
        src_name = id_to_name.get(src_id, src_id)
        tgt_name = id_to_name.get(tgt_id, tgt_id)

        if fk_id not in grouped:
            grouped[fk_id] = {
                "foreign_key_id": fk_id,
                "source_table_id": src_id,
                "source_table_name": src_name,
                "target_table_id": tgt_id,
                "target_table_name": tgt_name,
                "column_mappings": [],
            }

        for mapping in getattr(fk, "columns", None) or []:
            grouped[fk_id]["column_mappings"].append(
                {
                    "source_column_name": str(mapping.source_column_name),
                    "target_column_name": str(mapping.target_column_name),
                }
            )

    foreign_keys.extend(grouped.values())

    return {
        "tables": tables,
        "foreign_keys": foreign_keys,
        "table_count": len(tables),
        "foreign_key_count": len(foreign_keys),
    }


print("Celonis extraction helpers loaded.")

Celonis extraction helpers loaded.


## 4 — Connect to Celonis

Skipped when `SKIP_CELONIS_EXTRACTION = True`.

In [6]:
celonis = None
km = None

if not SKIP_CELONIS_EXTRACTION:
    from pycelonis import get_celonis

    _base_url = CELONIS_URL.rstrip("/")

    celonis = get_celonis(
        base_url=_base_url,
        api_token=CELONIS_API_TOKEN,
        key_type=CELONIS_KEY_TYPE,
        permissions=False,
        retries=3,
        delay=2,
    )
    print(f"Connected to {_base_url}")

    # Skip listing all spaces — that call is often blocked on corporate networks.
    # Go directly to the configured Space / Package / Knowledge Model.
    try:
        space = celonis.studio.get_space(SPACE_ID)
        package = space.get_package(PACKAGE_ID)
        km = package.get_knowledge_model(KNOWLEDGE_MODEL_ID)
        print(f"Space:           {getattr(space, 'name', SPACE_ID)}")
        print(f"Package:         {getattr(package, 'name', PACKAGE_ID)}")
        print(f"Knowledge Model: {getattr(km, 'name', KNOWLEDGE_MODEL_ID)}")
    except Exception as exc:
        raise RuntimeError(
            "Could not open the configured Space / Package / Knowledge Model. "
            "Check SPACE_ID, PACKAGE_ID, KNOWLEDGE_MODEL_ID in .env and that your "
            "API token has Studio access. Original error: "
            f"{type(exc).__name__}: {exc}"
        ) from exc
else:
    print("Skipping Celonis connection (offline mode).")

c:\Users\abcom\Desktop\ABI_Code_Migration_POC\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Connected to https://abinbev-prod.eu-3.celonis.cloud
Space:           Default
Package:         Global E2E
Knowledge Model: SAZ OTC - KM v2


## 5 — Extract KPIs and data model

In offline mode, place `extracted_kpis.json` and `data_model_metadata.json` under `OFFLINE_DIR`.

In [7]:
kpi_rows: List[Dict[str, Any]] = []
dm_metadata: Dict[str, Any] = {"tables": [], "foreign_keys": []}

if not SKIP_CELONIS_EXTRACTION:
    assert km is not None and celonis is not None

    kpi_rows = extract_kpis_from_knowledge_model(km)
    print(f"Extracted {len(kpi_rows)} KPI rows from KM '{KNOWLEDGE_MODEL_ID}'")

    dm_metadata = extract_data_model_metadata(celonis, DATA_MODEL_ID)
    print(
        f"Data model: {dm_metadata.get('table_count', 0)} tables, "
        f"{dm_metadata.get('foreign_key_count', 0)} foreign-key groups"
    )

    OFFLINE_DIR.mkdir(parents=True, exist_ok=True)
    (OFFLINE_DIR / "extracted_kpis.json").write_text(
        json.dumps(kpi_rows, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    (OFFLINE_DIR / "data_model_metadata.json").write_text(
        json.dumps(dm_metadata, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    print(f"Cached raw extracts to {OFFLINE_DIR.resolve()}")
else:
    kpi_path = OFFLINE_DIR / "extracted_kpis.json"
    dm_path = OFFLINE_DIR / "data_model_metadata.json"
    if not kpi_path.is_file():
        raise FileNotFoundError(f"Missing offline KPI file: {kpi_path}")
    kpi_rows = json.loads(kpi_path.read_text(encoding="utf-8"))
    if dm_path.is_file():
        dm_metadata = json.loads(dm_path.read_text(encoding="utf-8"))
    print(f"Offline mode: {len(kpi_rows)} KPI rows, {len(dm_metadata.get('tables', []))} tables")

Extracted 215 KPI rows from KM '9c910032-1f6d-4e49-ab70-d73a226af3a4'
Data model: 29 tables, 27 foreign-key groups
Cached raw extracts to C:\Users\abcom\Desktop\ABI_Code_Migration_POC\kpi_discovery_offline


## 6 — Build and write output files

In [8]:
kpis_out, dependents_out, connectivity_out = build_output_payloads(kpi_rows, dm_metadata)

written_paths = {}

# JSON outputs
for filename, payload in {
    "kpis.json": kpis_out,
    "sap_activity_tables_connectivity.json": connectivity_out,
}.items():
    path = OUTPUT_DIR / filename
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    written_paths[filename] = path
    print(f"Wrote {path} ({len(payload)} records, {path.stat().st_size // 1024} KB)")

# Word output — one section per KPI (replaces kpi_dependents.json)
dependents_docx = OUTPUT_DIR / "kpi_dependents.docx"
write_kpi_dependents_word(dependents_out, dependents_docx)
written_paths["kpi_dependents.docx"] = dependents_docx
print(f"Wrote {dependents_docx} ({len(dependents_out)} KPI sections, {dependents_docx.stat().st_size // 1024} KB)")

meta = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "kpi_count": len(kpis_out),
    "kpis_with_dependents": sum(1 for row in dependents_out if row["has_dependents"] == "yes"),
    "table_count": len(connectivity_out),
    "files": {name: str(path) for name, path in written_paths.items()},
}
(OUTPUT_DIR / "_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
print(f"\nSummary: {meta['kpi_count']} KPIs, {meta['kpis_with_dependents']} with dependents, {meta['table_count']} tables")

Wrote kpi_discovery_output\kpis.json (215 records, 345 KB)
Wrote kpi_discovery_output\kpi_dependents.json (215 records, 102 KB)
Wrote kpi_discovery_output\sap_activity_tables_connectivity.json (27 records, 12 KB)

Summary: 215 KPIs, 71 with dependents, 27 tables


## 7 — Preview outputs

In [9]:
import pandas as pd

pd.DataFrame(kpis_out).head(10)

,kpi_id,kpi_name,pql
0,COUNT_TABLE__FREIGHT_TMS_MAPPING,Count Table Freight Tms Mapping,"COUNT_TABLE(""FREIGHT_TMS_MAPPING"")"
1,PROCESS_VARIANTS__SHIPMENT_ACTIVITIES__ACTIVIT...,Process Variants Shipment Activities Activity En,"SHORTENED(VARIANT(""SHIPMENT_ACTIVITIES"".""ACTIV..."
2,COUNT_TABLE__S4_DOCK_CMTD_T,Count Table S4 Dock Cmtd T,"COUNT_TABLE(""S4_DOCK_CMTD_T"")"
3,COUNT_TABLE__MARA,Count Table Mara,"COUNT_TABLE(""MARA"")"
4,PROCESS_VARIANTS__S4_TMS_ACTIVITIES__ACTIVITY_EN,Process Variants S4 Tms Activities Activity En,"SHORTENED(VARIANT(""S4_TMS_ACTIVITIES"".""ACTIVIT..."
5,FILTERED_COUNT,Filtered Count,COUNT(CASE WHEN {p1} THEN 0 ELSE NULL END)
6,TOTAL_THROUGHPUT_TIME_IN_DAYS__FREIGHT_ACTIVIT...,Total Throughput Time In Days Freight Activiti...,"AVG(CALC_THROUGHPUT(CASE_START TO CASE_END, RE..."
7,COUNT_TABLE__VBAP,# Sales Order Items,"COUNT_TABLE(""VBAP"")"
8,NUMBER_OF_PROCESS_VARIANTS__INVOICE_ACTIVITIES...,Number Of Process Variants Invoice Activities ...,"COUNT(DISTINCT SHORTENED(VARIANT(""INVOICE_ACTI..."
9,COUNT_TABLE__VBAK,# Sales Order,"COUNT_TABLE(""VBAK"")"


In [10]:
dep_df = pd.DataFrame(dependents_out)
dep_with = dep_df[dep_df["has_dependents"] == "yes"].copy()
dep_with["path_count"] = dep_with["dependency_paths"].apply(len)
dep_with[["kpi_id", "kpi_name", "has_dependents", "kpi_dependents", "path_count"]].head(15)

,kpi_id,kpi_name,has_dependents,kpi_dependents,path_count
54,kpi_so_automation_percentage,Order Creation,yes,[kpi_so_automation_flag],1
56,kpi_ftr_so_automation_percentage,No Manual Order Changes,yes,[kpi_ftr_so_automation_flag],1
57,kpi_touchless_order_flag,Touchless Order Flag,yes,"[kpi_so_automation_flag, kpi_ftr_so_automation...",4
58,kpi_touchless_order_percentage,Touchless Order,yes,"[kpi_coverage_order_creation, kpi_so_automatio...",8
60,kpi_goods_issue_automation_percentage,Goods Issue,yes,[kpi_goods_issue_automation_flag],1
62,kpi_invoice_automation_percentage,Invoice Creation,yes,[kpi_invoice_automation_flag],1
65,kpi_no_invoice_cancellation_percentage,No Invoice Cancellation,yes,[kpi_no_invoice_cancellation_flag],1
66,kpi_no_debit_credit_memo_percentage,No Debit/Credit Memo,yes,[kpi_no_debit_credit_memo_flag],1
68,kpi_shipment_creation_automation_percentage,Shipment Creation Automation,yes,[kpi_shipment_creation_automation_flag],1
70,kpi_ftr_invoice_automation_percentage,No Manual Invoice Changes,yes,[kpi_ftr_invoice_automation_flag],1


In [11]:
tables_df = pd.DataFrame(dependents_out)
tables_df["sap_count"] = tables_df["sap_tables_used"].apply(len)
tables_df["activity_count"] = tables_df["activity_tables_used"].apply(len)
tables_df[(tables_df["sap_count"] > 0) | (tables_df["activity_count"] > 0)][
    ["kpi_id", "kpi_name", "sap_tables_used", "activity_tables_used"]
].head(15)

,kpi_id,kpi_name,sap_tables_used,activity_tables_used
0,COUNT_TABLE__FREIGHT_TMS_MAPPING,Count Table Freight Tms Mapping,[FREIGHT_TMS_MAPPING],[]
1,PROCESS_VARIANTS__SHIPMENT_ACTIVITIES__ACTIVIT...,Process Variants Shipment Activities Activity En,[],[SHIPMENT_ACTIVITIES]
2,COUNT_TABLE__S4_DOCK_CMTD_T,Count Table S4 Dock Cmtd T,[S4_DOCK_CMTD_T],[]
3,COUNT_TABLE__MARA,Count Table Mara,[MARA],[]
4,PROCESS_VARIANTS__S4_TMS_ACTIVITIES__ACTIVITY_EN,Process Variants S4 Tms Activities Activity En,[],[S4_TMS_ACTIVITIES]
6,TOTAL_THROUGHPUT_TIME_IN_DAYS__FREIGHT_ACTIVIT...,Total Throughput Time In Days Freight Activiti...,[],[FREIGHT_ACTIVITIES]
7,COUNT_TABLE__VBAP,# Sales Order Items,[VBAP],[]
8,NUMBER_OF_PROCESS_VARIANTS__INVOICE_ACTIVITIES...,Number Of Process Variants Invoice Activities ...,[],[INVOICE_ACTIVITIES]
9,COUNT_TABLE__VBAK,# Sales Order,[VBAK],[]
10,COUNT_TABLE__ACTIVITIES,# Activities,[],[ACTIVITIES]


In [12]:
conn_df = pd.DataFrame(connectivity_out)
conn_df["join_count"] = conn_df["connections"].apply(len)
conn_df.sort_values("join_count", ascending=False).head(15)

,table_name,connections,join_count
17,SAZ_ECC_S4_COMBINED_VBAP_Final,[{'source': {'table': 'SAZ_ECC_S4_COMBINED_VBA...,5
22,SAZ_ECC_S4_COMBINED_VTTK,[{'source': {'table': 'SAZ_ECC_S4_COMBINED_VTT...,3
2,BRAZIL_LD_LEG_T_VIEW,"[{'source': {'table': 'BRAZIL_LD_LEG_T_VIEW', ...",3
12,SAZ_ECC_S4_COMBINED_LIKP,[{'source': {'table': 'SAZ_ECC_S4_COMBINED_LIK...,2
13,SAZ_ECC_S4_COMBINED_LIPS_Final,[{'source': {'table': 'SAZ_ECC_S4_COMBINED_LIP...,2
20,SAZ_ECC_S4_COMBINED_VBRK,[{'source': {'table': 'SAZ_ECC_S4_COMBINED_VBR...,2
16,SAZ_ECC_S4_COMBINED_VBAK,[{'source': {'table': 'SAZ_ECC_S4_COMBINED_VBA...,1
6,SAZ_ECC_S4_COMBINED_BKPF,[{'source': {'table': 'SAZ_ECC_S4_COMBINED_BKP...,1
10,SAZ_ECC_S4_COMBINED_KNA1_Final,[{'source': {'table': 'SAZ_ECC_S4_COMBINED_KNA...,1
5,PARITY_GRID,"[{'source': {'table': 'PARITY_GRID', 'columns'...",1


## 8 — Drill-down: single KPI

In [13]:
TARGET_KPI_ID = "kpi_touchless_order_percentage"

kpi_rec = next((row for row in kpis_out if row["kpi_id"] == TARGET_KPI_ID), None)
dep_rec = next((row for row in dependents_out if row["kpi_id"] == TARGET_KPI_ID), None)

print("=== kpis.json ===")
print(json.dumps(kpi_rec, indent=2))
print("\n=== kpi_dependents (preview) ===")
if dep_rec:
    preview = {**dep_rec, "dependency_paths": dep_rec["dependency_paths"][:5]}
    print(json.dumps(preview, indent=2))
    if len(dep_rec["dependency_paths"]) > 5:
        print(f"  ... and {len(dep_rec['dependency_paths']) - 5} more paths")
else:
    print("KPI not found in dependents output.")

=== kpis.json ===
{
  "kpi_id": "kpi_touchless_order_percentage",
  "kpi_name": "Touchless Order",
  "pql": "(\n((KPI(\"kpi_coverage_order_creation\")*COUNT_TABLE(\"VBAK\"))*KPI(\"kpi_so_automation_percentage\"))\n+\n((KPI(\"kpi_coverage_credit_check\")*COUNT_TABLE(\"VBAK\"))*KPI(\"kpi_credit_check_percentage\"))\n+\n((KPI(\"kpi_coverage_order_changes\")*COUNT_TABLE(\"VBAK\"))*KPI(\"kpi_ftr_so_automation_percentage\"))\n+\n((KPI(\"kpi_coverage_quotations\")*COUNT_TABLE(\"VBAK\"))*KPI(\"kpi_quotations_percentage\"))\n)\n/\n(\n(KPI(\"kpi_coverage_order_creation\")*COUNT_TABLE(\"VBAK\"))\n+\nCOUNT_TABLE(\"VBAK\")\n+\n(KPI(\"kpi_coverage_order_changes\")*COUNT_TABLE(\"VBAK\"))\n+\n(KPI(\"kpi_coverage_quotations\")*COUNT_TABLE(\"VBAK\"))\n)"
}

=== kpi_dependents.json ===
{
  "kpi_id": "kpi_touchless_order_percentage",
  "kpi_name": "Touchless Order",
  "has_dependents": "yes",
  "kpi_dependents": [
    "kpi_coverage_order_creation",
    "kpi_so_automation_percentage",
    "kpi_coverage_cre